In [26]:
import numpy as np
from ket import *

# ==========================
# Optimizer
# ==========================

class Optimize:

    # Mean Squared Error loss function
    def loss(self, z, y):
        p = (z + 1) / 2
        return np.mean((p - y) ** 2)

    # Finite difference gradient estimation
    def gradient(self, qnn, x, weights, y):

        grad = np.zeros(len(weights))

        z = qnn.run_circuit(x, weights)
        p = (z + 1) / 2

        dL_dz = p - y

        for i in range(len(weights)):

            w_plus = weights.copy()
            w_minus = weights.copy()

            w_plus[i] += np.pi / 2
            w_minus[i] -= np.pi / 2

            z_plus = qnn.run_circuit(x, w_plus)
            z_minus = qnn.run_circuit(x, w_minus)

            dz_dtheta = (z_plus - z_minus) / 2

            grad[i] = np.sum(dL_dz * dz_dtheta)

        return grad


# ==========================
# Quantum Neural Network
# ==========================

class QNN:

    def __init__(self, lines, ansatz):
        self.lines = lines
        self.ansatz = ansatz

    # Simple angle encoding using RY rotations
    def angle_encoding(self, parameters, qubits):
        for i in range(len(parameters)):
            RY(parameters[i], qubits[i])

    # Compute expectation value of Z for each qubit from measurement counts
    def expectation_z(self, counts):

        total_shots = sum(counts.values())

        exp_z = np.zeros(self.lines)

        for state, count in counts.items():

            bitstring = format(state, f"0{self.lines}b")

            for q, bit in enumerate(bitstring):

                exp_z[q] += count * (
                    1 if bit == "0" else -1
                )

        return exp_z / total_shots

    # Run the quantum circuit with given input and weights, then return the expectation value of Z
    def run_circuit(self, x, weights):

        process = Process()

        qubits = process.alloc(self.lines)

        self.angle_encoding(x, qubits)

        self.ansatz(weights, qubits)

        counts = sample(qubits, shots=1000).get()

        return self.expectation_z(counts)

    # Forward pass through the QNN
    def forward(self, x, weights):

        return self.run_circuit(x, weights)

    # Predict probabilities by mapping expectation values to [0, 1]
    def predict(self, x, weights):

        z = self.forward(x, weights)

        return (z + 1) / 2

    # Single training step using finite difference gradient estimation
    def train_step(self, x, y, weights, lr):

        optimizer = Optimize()

        grad = optimizer.gradient(self, x, weights, y)

        weights -= lr * grad

        loss = optimizer.loss(self.forward(x, weights), y)

        return weights, loss

    # Fit the model to the data using gradient descent
    def fit(self, x, y, weights, epochs=100, lr=0.01):

        for epoch in range(epochs):

            weights, loss = self.train_step(x, y, weights, lr)

            print(
                f"Epoch {epoch+1}/{epochs} "
                f"| Loss = {loss:.6f}"
            )

        return weights

In [33]:
import numpy as np

# ==========================
# Dataset
# ==========================

N = 200

X = np.random.uniform(
    0,
    np.pi,
    size=(N, 2)
)

Y = np.array([
    [1] if x[0] > x[1] else [0]
    for x in X
])

# embaralha
idx = np.random.permutation(N)

X = X[idx]
Y = Y[idx]

# treino/teste
split = int(0.75 * N)

X_train = X[:split]
Y_train = Y[:split]

X_test = X[split:]
Y_test = Y[split:]

print("Treino:", len(X_train))
print("Teste:", len(X_test))

# ==========================
# Ansatz
# ==========================

def hec_ansatz(weights, qubits):

    for i in range(len(qubits)):
        RY(weights[i], qubits[i])

    for i in range(len(qubits) - 1):
        CNOT(qubits[i], qubits[i + 1])

# ==========================
# Modelo
# ==========================

qnn = QNN(
    lines=2,
    ansatz=hec_ansatz
)

weights = np.random.randn(2)

# ==========================
# Treinamento
# ==========================

epochs = 150

for epoch in range(epochs):

    total_loss = 0

    for x, y in zip(X_train, Y_train):

        weights, loss = qnn.train_step(
            x=x,
            y=y,
            weights=weights,
            lr=0.05
        )

        total_loss += loss

    avg_loss = total_loss / len(X_train)

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch}/{epochs} "
            f"| Loss = {avg_loss:.6f}"
        )

print("\nPesos finais:")
print(weights)

# ==========================
# Avaliação
# ==========================

correct = 0

for x, y in zip(X_test, Y_test):

    pred = qnn.predict(x, weights)[0]

    pred_class = 1 if pred > 0.5 else 0

    if pred_class == y[0]:
        correct += 1

accuracy = correct / len(X_test)

print("\nAcurácia:", accuracy)

# ==========================
# Exemplos
# ==========================

print("\nAlgumas previsões:\n")

for i in range(50):

    pred = qnn.predict(
        X_test[i],
        weights
    )[0]

    pred_class = 1 if pred > 0.5 else 0

    print(
        f"x={X_test[i]} | "
        f"real={Y_test[i][0]} | "
        f"prob={pred:.3f} | "
        f"classe={pred_class}"
    )

Treino: 150
Teste: 50
Epoch 0/150 | Loss = 0.352374
Epoch 10/150 | Loss = 0.158582
Epoch 20/150 | Loss = 0.160632
Epoch 30/150 | Loss = 0.160360
Epoch 40/150 | Loss = 0.157908
Epoch 50/150 | Loss = 0.159313
Epoch 60/150 | Loss = 0.159016
Epoch 70/150 | Loss = 0.158961
Epoch 80/150 | Loss = 0.158364
Epoch 90/150 | Loss = 0.158530
Epoch 100/150 | Loss = 0.159529
Epoch 110/150 | Loss = 0.159287
Epoch 120/150 | Loss = 0.158857
Epoch 130/150 | Loss = 0.159479
Epoch 140/150 | Loss = 0.159689

Pesos finais:
[3.61808876 5.10808933]

Acurácia: 0.72

Algumas previsões:

x=[1.26340496 2.28152601] | real=0 | prob=0.568 | classe=1
x=[1.0394125  2.25611922] | real=0 | prob=0.476 | classe=0
x=[1.81111295 0.92912401] | real=1 | prob=0.816 | classe=1
x=[3.02367271 3.01711489] | real=1 | prob=0.966 | classe=1
x=[0.68744539 3.05938058] | real=0 | prob=0.287 | classe=0
x=[2.90422361 2.91539316] | real=0 | prob=0.986 | classe=1
x=[1.02977008 0.27125253] | real=1 | prob=0.491 | classe=0
x=[2.45628782 1.9464